In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Dom.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Amount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Amount,concept:name,org:resource,org:role,time_delta
0,declaration 100000,2018-01-30 09:20:07,600.844116,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 100000,2018-02-07 09:58:46,600.844116,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,693519.0
2,declaration 100000,2018-02-08 10:59:05,600.844116,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,90019.0
3,declaration 100000,2018-02-09 12:42:49,600.844116,Request Payment,SYSTEM,UNDEFINED,92624.0
4,declaration 100000,2018-02-12 17:31:20,600.844116,Payment Handled,SYSTEM,UNDEFINED,276511.0
5,declaration 100005,2018-01-30 09:38:54,35.133686,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
6,declaration 100005,2018-01-30 09:38:57,35.133686,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0
7,declaration 100005,2018-01-30 10:04:10,35.133686,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,1513.0
8,declaration 100005,2018-01-31 12:45:18,35.133686,Request Payment,SYSTEM,UNDEFINED,96068.0
9,declaration 100005,2018-02-01 17:31:17,35.133686,Payment Handled,SYSTEM,UNDEFINED,103559.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Amount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [2.00, 284911.00]                        42289.5000 quantile_derived    
case:Amount                    continuous     case     yes    [6.91, 219.03]                           25.3885    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
org:role                       categorical    event    ye

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[]

In [13]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by MISSING'},
 {'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Dom-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/220 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 142992,4,1,0,0.294198,0.225895,0.362500,0.435714,0.272727,...,1.054760,0.272727,0.000027,0.0000,0.000054,0.142857,0.639148,0.639148,0.999998,0.999998
1,0,declaration 115669,4,1,0,0.342766,0.423033,0.262500,0.407143,0.272727,...,0.947288,0.272727,0.000000,0.0000,0.000000,0.000000,0.674561,0.674561,0.999999,0.999999
2,0,declaration 138710,4,1,0,0.306472,0.325443,0.287500,0.385714,0.272727,...,0.898285,0.272727,0.000000,0.0000,0.000000,0.000000,0.625557,0.625557,0.999998,0.999998
3,0,declaration 141310,4,1,0,0.358969,0.417937,0.300000,0.400000,0.272727,...,0.947223,0.272727,0.000000,0.0000,0.000000,0.000000,0.674496,0.674496,0.999999,0.999999
4,0,declaration 113587,5,1,0,0.334945,0.332391,0.337500,0.471429,0.230769,...,0.803474,0.230769,0.000000,0.0000,0.000000,0.000000,0.572704,0.572704,0.999996,0.999996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,20,declaration 124561,11,1,0,0.386917,0.373833,0.400000,0.502000,0.400000,...,0.653880,0.400000,0.093880,0.1875,0.000261,0.160000,0.000000,0.628985,0.000000,0.999998
148,20,declaration 134394,11,1,0,0.423076,0.486777,0.359375,0.572000,0.400000,...,0.826352,0.400000,0.106352,0.0000,0.212705,0.320000,0.000000,0.567507,0.000000,0.999995
149,20,declaration 129484,11,1,0,0.387637,0.359650,0.415625,0.498000,0.396000,...,0.391250,0.320000,0.031250,0.0625,0.000000,0.040000,0.000000,0.400075,0.000000,0.000000
150,20,declaration 126499,11,1,0,0.432386,0.389772,0.475000,0.564000,0.400000,...,0.471250,0.400000,0.031250,0.0625,0.000000,0.040000,0.000000,0.149892,0.000000,0.000000


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/420 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 142992,4,2,0,0.532543,0.490087,0.575000,0.5750,0.021429,...,0.840967,0.000000,0.340967,0.5000,0.181934,0.50,0.000000,0.477160,0.0,0.500000
1,0,declaration 115669,4,2,0,0.674296,0.623592,0.725000,0.7625,0.128571,...,0.896035,0.000000,0.396035,0.5000,0.292070,0.50,0.000000,0.478308,0.0,0.500000
2,0,declaration 138710,4,2,0,0.490508,0.456016,0.525000,0.5875,0.171429,...,0.886817,0.000000,0.384780,0.5000,0.269560,0.50,0.002037,0.479285,0.0,0.500000
3,0,declaration 141310,4,2,0,0.775111,0.750223,0.800000,0.8375,0.021429,...,0.897201,0.000000,0.397201,0.5000,0.294402,0.50,0.000000,0.478387,0.0,0.500000
4,1,declaration 92054,4,2,0,0.370910,0.416820,0.325000,0.4875,0.142857,...,0.142857,0.142857,0.000000,0.0000,0.000000,0.00,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,40,declaration 124561,11,3,0,0.441104,0.488458,0.393750,0.6000,0.193548,...,0.566632,0.193548,0.053083,0.0000,0.106167,0.32,0.000000,0.841399,0.0,0.999999
263,40,declaration 134394,11,3,0,0.428165,0.418830,0.437500,0.6120,0.193548,...,0.600574,0.193548,0.087026,0.0000,0.174052,0.32,0.000000,0.858809,0.0,0.999999
264,40,declaration 129484,11,3,0,0.406417,0.381583,0.431250,0.5400,0.193548,...,0.313993,0.193548,0.040444,0.0000,0.080888,0.08,0.000000,0.801488,0.0,0.666667
265,40,declaration 126499,11,3,0,0.361868,0.358112,0.365625,0.4860,0.193548,...,0.344884,0.193548,0.031336,0.0625,0.000172,0.12,0.000000,0.678523,0.0,0.666667


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()